# Week 4, Day 6: Forge: Week 4 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w004/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", np.linalg.norm([1, 1, 1, 1]))
a, b = np.array([2, -1]), np.array([1, 2])
print("2.", a @ b, "so the angle is", np.degrees(np.arccos(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))), "degrees")
x, y = np.array([1, 2, 3.0]), np.array([10, 20, 30.0])
print("3.", x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(1986)
n_neurons, n_dirs, n_trials, T = 80, 8, 20, 0.5
dirs = np.radians(np.arange(0, 360, 45))                     # (8,)
true_pd = rng.uniform(0, 2 * np.pi, n_neurons)
b0 = rng.uniform(10, 30, n_neurons)
k = b0 * rng.uniform(0.3, 0.9, n_neurons)                   # modulation never exceeds baseline
rate = b0[:, None] + k[:, None] * np.cos(dirs[None, :] - true_pd[:, None])     # (80, 8)
counts = rng.poisson(rate[:, :, None] * T, size=(n_neurons, n_dirs, n_trials))  # (80, 8, 20)
print(counts.shape)

### Exercise: Part A: preferred directions from training trials

Split trials into `train` (the first 10) and `test` (the last 10). From `train`, compute each neuron's mean rate per direction (Hz), then estimate `base_est` (mean over directions), `pd_est` (the angle of the rate-weighted sum of direction unit vectors, in radians) and `k_est` (modulation depth: twice the length of that sum divided by the number of directions, from Day 3's proof). No loop over neurons.

In [ ]:
train, test = ..., ...
mean_rate = ...                       # (80, 8) in Hz
base_est = ...
x = ...                               # (80,) rate-weighted cos sum
y = ...
pd_est = np.arctan2(y, x) % (2 * np.pi)
k_est = ...
print(np.degrees(pd_est[:5]).round(0), np.degrees(true_pd[:5]).round(0))

In [ ]:
import numpy as np
def test_split():
    "train and test each use 10 trials"
    assert train.shape == (80, 8, 10) and test.shape == (80, 8, 10)

def test_estimates():
    "pd_est, base_est and k_est have one value per neuron"
    assert pd_est.shape == (80,) and base_est.shape == (80,) and k_est.shape == (80,)

def test_accuracy():
    "preferred directions are recovered to within 20 degrees on average"
    err = np.degrees(np.abs(np.angle(np.exp(1j * (pd_est - true_pd)))))
    assert err.mean() < 20, f"mean error {err.mean():.1f} degrees"

test_split()
test_estimates()
test_accuracy()
print("All checks passed")

### Exercise: Part B: decode the held-out trials

For every test trial, compute the population vector using the estimates from Part A: weights $(r_i - \hat b_i)/\hat k_i$ on the unit vectors of `pd_est`. Store the decoded directions in degrees in `decoded` (shape 8 by 10) and the mean absolute angular error in `mean_err`. Why must the preferred directions come from different trials than the ones you decode?

In [ ]:
test_rate = test / T                                         # (80, 8, 10)
w = ...                                                      # weights, same shape
px = ...                                                     # sum over neurons, shape (8, 10)
py = ...
decoded = np.degrees(np.arctan2(py, px)) % 360
mean_err = ...
print(f"mean error on held-out trials: {mean_err:.1f} degrees")

In [ ]:
import numpy as np
def test_decoded():
    "decoded has one direction per direction and test trial"
    assert decoded.shape == (8, 10)

def test_error():
    "mean decoding error is under 15 degrees"
    assert mean_err < 15, f"mean error {mean_err:.1f} degrees"
    true = np.degrees(dirs)[:, None]
    e = np.abs((decoded - true + 180) % 360 - 180)
    assert abs(e.mean() - mean_err) < 1e-9

test_decoded()
test_error()
print("All checks passed")

### Exercise: Part C: the geometry of the code

Using the training mean rates, build `D`, the 8 by 8 representational dissimilarity matrix ($1 - r$ between the population patterns for each pair of directions). Show it as an image. What shape does it have, and what does it say about how the population arranges the eight directions?

In [ ]:
D = ...
plt.figure(figsize=(3.6, 3))
plt.imshow(D, cmap="viridis"); plt.colorbar(label="1 - r")
plt.xticks(range(8), np.arange(0, 360, 45)); plt.yticks(range(8), np.arange(0, 360, 45))
plt.xlabel("direction (deg)"); plt.ylabel("direction (deg)"); plt.show()

In [ ]:
import numpy as np
def test_rdm():
    "D is the 8 by 8 matrix of one minus correlation between direction patterns"
    assert D.shape == (8, 8) and np.allclose(np.diag(D), 0)

def test_ring():
    "neighbouring directions are more similar than opposite ones"
    assert D[0, 1] < D[0, 4] and D[2, 3] < D[2, 6]

test_rdm()
test_ring()
print("All checks passed")